# AI Sentiment vs Shipping Traffic Analysis
Correlating real news sentiment with shipping disruptions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

plt.rcParams['figure.figsize'] = (20, 10)
plt.rcParams['font.size'] = 11

## Load Shipping Data

In [ ]:
arrivals_df = pd.read_csv('Data/Portwatch_Shipment_Data/arrivals-of-ships.csv')
arrivals_df['DateTime'] = pd.to_datetime(arrivals_df['DateTime'])
arrivals_df.set_index('DateTime', inplace=True)
arrivals_df['Total'] = arrivals_df[['Container', 'Dry Bulk', 'General Cargo', 'Roll-on/roll-off', 'Tanker']].sum(axis=1)

arrivals_df.head()

## Load AI-Analyzed Sentiment Data

In [ ]:
sentiment_df = pd.read_csv('Data/crisis_sentiment_analysis.csv')
sentiment_df['date'] = pd.to_datetime(sentiment_df['date'])

sentiment_daily = pd.read_csv('Data/crisis_sentiment_daily.csv')
sentiment_daily['date'] = pd.to_datetime(sentiment_daily['date'])

print(f"Total articles analyzed: {len(sentiment_df)}")
print(f"Date range: {sentiment_df['date'].min()} to {sentiment_df['date'].max()}")
print(f"Crisis events: {sentiment_df['crisis_event'].nunique()}")

sentiment_df.head(10)

In [ ]:
sentiment_df.info()

## Sentiment Statistics by Crisis

In [ ]:
sentiment_stats = sentiment_df.groupby('crisis_event').agg({
    'sentiment': ['count', 'mean', 'std', 'min', 'max'],
    'confidence': 'mean',
    'used_full_article': 'sum',
    'article_length': 'mean'
}).round(3)

sentiment_stats.columns = ['Articles', 'Avg_Sentiment', 'Std', 'Min', 'Max', 'Confidence', 'Full_Articles', 'Avg_Length']
sentiment_stats = sentiment_stats.sort_values('Avg_Sentiment')
sentiment_stats

## Merge Sentiment with Shipping Data

In [ ]:
merged_df = arrivals_df.reset_index()
merged_df['date'] = merged_df['DateTime'].dt.date
merged_df['date'] = pd.to_datetime(merged_df['date'])

sentiment_daily['date'] = sentiment_daily['date'].dt.date
sentiment_daily['date'] = pd.to_datetime(sentiment_daily['date'])

merged_df = merged_df.merge(
    sentiment_daily[['date', 'sentiment_mean', 'risk_level', 'article_count', 'crisis_event']], 
    on='date', 
    how='left'
)

merged_df.set_index('DateTime', inplace=True)

merged_df[merged_df['sentiment_mean'].notna()].head(10)

## Correlation Analysis

In [ ]:
correlation_data = merged_df[['Tanker', 'Container', 'Dry Bulk', 'Total', 'sentiment_mean']].dropna()
correlation_matrix = correlation_data.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={'shrink': 0.8})
plt.title('Correlation: AI Sentiment vs Ship Traffic', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

correlation_matrix

## Tanker Traffic vs AI Sentiment Over Time

In [ ]:
fig, ax1 = plt.subplots(figsize=(24, 10))
ax2 = ax1.twinx()

ax1.plot(merged_df.index, merged_df['Tanker'], linewidth=2.5, color='darkred', 
         label='Tanker Traffic (Oil Carriers)', alpha=0.8, zorder=1)

sentiment_data = merged_df[merged_df['sentiment_mean'].notna()]
scatter = ax2.scatter(sentiment_data.index, sentiment_data['sentiment_mean'], 
                     c=sentiment_data['sentiment_mean'], cmap='RdYlGn', 
                     s=150, alpha=0.7, edgecolors='black', linewidth=1, zorder=2,
                     vmin=-1, vmax=0.5)

ax2.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.3)
ax2.axhline(y=-0.5, color='red', linestyle=':', linewidth=1.5, alpha=0.5, label='High Risk Threshold')

ax1.set_ylabel('Daily Tanker Arrivals', fontsize=14, color='darkred', fontweight='bold')
ax2.set_ylabel('AI Sentiment Score\n(-1=Extreme Negative, +1=Positive)', fontsize=14, color='green', fontweight='bold')
ax1.set_xlabel('Date', fontsize=14)
ax1.set_title('Tanker Traffic vs AI-Analyzed News Sentiment (2019-2026)', fontsize=18, fontweight='bold', pad=20)

ax1.legend(loc='upper left', fontsize=12)
ax2.legend(loc='upper right', fontsize=12)
ax1.grid(True, alpha=0.3)

cbar = plt.colorbar(scatter, ax=ax2, pad=0.02)
cbar.set_label('Sentiment Score', fontsize=12)

plt.tight_layout()
plt.show()

## Total Traffic vs Sentiment

In [ ]:
fig, ax1 = plt.subplots(figsize=(24, 10))
ax2 = ax1.twinx()

ax1.plot(merged_df.index, merged_df['Total'], linewidth=2.5, color='navy', 
         label='Total Ship Traffic', alpha=0.8, zorder=1)

sentiment_data = merged_df[merged_df['sentiment_mean'].notna()]
scatter = ax2.scatter(sentiment_data.index, sentiment_data['sentiment_mean'], 
                     c=sentiment_data['sentiment_mean'], cmap='RdYlGn', 
                     s=150, alpha=0.7, edgecolors='black', linewidth=1, zorder=2,
                     vmin=-1, vmax=0.5)

ax2.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.3)
ax2.axhline(y=-0.5, color='red', linestyle=':', linewidth=1.5, alpha=0.5)

ax1.set_ylabel('Daily Total Arrivals', fontsize=14, color='navy', fontweight='bold')
ax2.set_ylabel('AI Sentiment Score', fontsize=14, color='green', fontweight='bold')
ax1.set_xlabel('Date', fontsize=14)
ax1.set_title('Total Ship Traffic vs AI-Analyzed News Sentiment', fontsize=18, fontweight='bold', pad=20)

ax1.legend(loc='upper left', fontsize=12)
ax1.grid(True, alpha=0.3)

cbar = plt.colorbar(scatter, ax=ax2, pad=0.02)
cbar.set_label('Sentiment Score', fontsize=12)

plt.tight_layout()
plt.show()

## Sentiment Scores by Crisis Event

In [ ]:
fig, ax = plt.subplots(figsize=(18, 10))

colors = ['darkred' if val < -0.5 else 'red' if val < -0.3 else 'orange' if val < 0 else 'green' 
          for val in sentiment_stats['Avg_Sentiment']]

bars = ax.barh(range(len(sentiment_stats)), sentiment_stats['Avg_Sentiment'], 
               color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)

ax.set_yticks(range(len(sentiment_stats)))
ax.set_yticklabels(sentiment_stats.index, fontsize=12)
ax.axvline(x=0, color='black', linestyle='-', linewidth=2)
ax.axvline(x=-0.5, color='red', linestyle='--', linewidth=1.5, alpha=0.5, label='High Risk')
ax.set_title('Average AI Sentiment Score by Crisis Event', fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Average Sentiment Score (-1=Extreme Negative, +1=Positive)', fontsize=14)
ax.grid(True, alpha=0.3, axis='x')
ax.legend(fontsize=12)

for i, (idx, row) in enumerate(sentiment_stats.iterrows()):
    ax.text(row['Avg_Sentiment'], i, 
            f"  {row['Avg_Sentiment']:.3f} ({int(row['Articles'])} articles)", 
            va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## Risk Level Distribution by Crisis

In [ ]:
risk_distribution = sentiment_df.groupby(['crisis_event', 'risk_level']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(18, 10))

risk_colors = {'low': 'green', 'medium': 'yellow', 'high': 'orange', 'critical': 'red', 'unknown': 'gray'}
risk_distribution.plot(kind='barh', stacked=True, ax=ax, 
                      color=[risk_colors.get(col, 'gray') for col in risk_distribution.columns],
                      edgecolor='black', linewidth=0.8)

ax.set_title('AI Risk Level Assessment by Crisis Event', fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Number of Articles', fontsize=14)
ax.set_ylabel('Crisis Event', fontsize=14)
ax.legend(title='AI Risk Level', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=12)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

risk_distribution

## Sentiment Timeline by Event

In [ ]:
fig, ax = plt.subplots(figsize=(22, 12))

for event in sentiment_df['crisis_event'].unique():
    event_data = sentiment_df[sentiment_df['crisis_event'] == event]
    ax.scatter(event_data['date'], event_data['sentiment'], 
              label=event, alpha=0.7, s=120, edgecolors='black', linewidth=0.5)

ax.axhline(y=0, color='black', linestyle='--', linewidth=1.5, alpha=0.5, label='Neutral')
ax.axhline(y=-0.5, color='red', linestyle=':', linewidth=2, alpha=0.5, label='High Risk')
ax.axhline(y=-0.8, color='darkred', linestyle=':', linewidth=2, alpha=0.5, label='Critical Risk')

ax.set_title('AI Sentiment Scores Over Time by Crisis Event', fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Date', fontsize=14)
ax.set_ylabel('Sentiment Score (-1=Extreme Negative, +1=Positive)', fontsize=14)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Most Negative Articles

In [ ]:
most_negative = sentiment_df.nsmallest(15, 'sentiment')[[
    'date', 'crisis_event', 'title', 'sentiment', 'risk_level', 'confidence', 'key_factors'
]]
most_negative

## Impact Analysis: Sentiment vs Traffic Drop

In [ ]:
impact_analysis = []

for event in sentiment_df['crisis_event'].unique():
    event_sentiment = sentiment_df[sentiment_df['crisis_event'] == event]
    
    if len(event_sentiment) > 0:
        start_date = event_sentiment['date'].min()
        end_date = event_sentiment['date'].max()
        
        event_traffic = merged_df[(merged_df.index >= start_date) & (merged_df.index <= end_date)]
        
        pre_event_start = start_date - timedelta(days=30)
        pre_event_traffic = merged_df[(merged_df.index >= pre_event_start) & (merged_df.index < start_date)]
        
        if len(event_traffic) > 0 and len(pre_event_traffic) > 0:
            impact_analysis.append({
                'Event': event,
                'Avg_Sentiment': event_sentiment['sentiment'].mean(),
                'Min_Sentiment': event_sentiment['sentiment'].min(),
                'Pre_Tanker_Avg': pre_event_traffic['Tanker'].mean(),
                'Event_Tanker_Avg': event_traffic['Tanker'].mean(),
                'Tanker_Drop_Pct': ((event_traffic['Tanker'].mean() - pre_event_traffic['Tanker'].mean()) / 
                                   pre_event_traffic['Tanker'].mean() * 100),
                'Pre_Total_Avg': pre_event_traffic['Total'].mean(),
                'Event_Total_Avg': event_traffic['Total'].mean(),
                'Total_Drop_Pct': ((event_traffic['Total'].mean() - pre_event_traffic['Total'].mean()) / 
                                  pre_event_traffic['Total'].mean() * 100)
            })

impact_df = pd.DataFrame(impact_analysis).round(2)
impact_df = impact_df.sort_values('Avg_Sentiment')
impact_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

axes[0].scatter(impact_df['Avg_Sentiment'], impact_df['Tanker_Drop_Pct'], 
               s=200, alpha=0.6, c=impact_df['Avg_Sentiment'], cmap='RdYlGn', 
               edgecolors='black', linewidth=2)
axes[0].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[0].axvline(x=0, color='black', linestyle='--', linewidth=1)
axes[0].set_title('Sentiment vs Tanker Traffic Impact', fontsize=16, fontweight='bold')
axes[0].set_xlabel('Average AI Sentiment Score', fontsize=12)
axes[0].set_ylabel('Tanker Traffic Change (%)', fontsize=12)
axes[0].grid(True, alpha=0.3)

for _, row in impact_df.iterrows():
    axes[0].annotate(row['Event'][:20], (row['Avg_Sentiment'], row['Tanker_Drop_Pct']), 
                    fontsize=8, alpha=0.7)

axes[1].scatter(impact_df['Avg_Sentiment'], impact_df['Total_Drop_Pct'], 
               s=200, alpha=0.6, c=impact_df['Avg_Sentiment'], cmap='RdYlGn', 
               edgecolors='black', linewidth=2)
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[1].axvline(x=0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Sentiment vs Total Traffic Impact', fontsize=16, fontweight='bold')
axes[1].set_xlabel('Average AI Sentiment Score', fontsize=12)
axes[1].set_ylabel('Total Traffic Change (%)', fontsize=12)
axes[1].grid(True, alpha=0.3)

for _, row in impact_df.iterrows():
    axes[1].annotate(row['Event'][:20], (row['Avg_Sentiment'], row['Total_Drop_Pct']), 
                    fontsize=8, alpha=0.7)

plt.tight_layout()
plt.show()

## Key Findings

In [ ]:
findings = {
    'Total Articles Analyzed': len(sentiment_df),
    'Crisis Events Covered': sentiment_df['crisis_event'].nunique(),
    'Date Range': f"{sentiment_df['date'].min().date()} to {sentiment_df['date'].max().date()}",
    'Most Negative Event': sentiment_stats.index[0],
    'Most Negative Avg Sentiment': sentiment_stats['Avg_Sentiment'].iloc[0],
    'Correlation (Sentiment vs Tanker)': correlation_matrix.loc['sentiment_mean', 'Tanker'].round(3),
    'Correlation (Sentiment vs Total)': correlation_matrix.loc['sentiment_mean', 'Total'].round(3),
    'Avg Confidence Score': sentiment_df['confidence'].mean().round(3),
    'Full Articles Analyzed': sentiment_df['used_full_article'].sum()
}

findings_df = pd.DataFrame([findings]).T
findings_df.columns = ['Value']
findings_df